# Regresión logística

1. Por qué los mínimos cuadrados fallan para clasificar.
2. Sigmoide, odds y log-odds.
3. Verosimilitud de Bernoulli y su gradiente, derivados y verificados.
4. Ajuste: descenso por gradiente y Newton/IRLS desde cero.
5. Convexidad, comprobada sobre la hessiana.
6. Datos separables: el estimador de máxima verosimilitud **no existe**.
7. Regularización, validación cruzada e interpretación por odds ratios.
8. Calibración y softmax multiclase.

**Notación (Bishop).** $\mathbf{x}_n$ entradas, $t_n \in \{0,1\}$ etiquetas,
$\mathbf{w}$ parámetros, $y_n = \sigma(\mathbf{w}^{\top}\mathbf{x}_n)$ la probabilidad predicha.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (6, 3.6), "axes.grid": True,
                     "grid.alpha": 0.25, "figure.dpi": 110})
np.set_printoptions(precision=4, suppress=True)

# sklearn >= 1.8 avisa de que `penalty=` está en camino de desaparecer en favor de
# `l1_ratio`. Seguimos usando `penalty=` porque funciona en todas las versiones que
# se encuentran en la práctica (y en Colab); solo silenciamos el aviso.
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Por qué no mínimos cuadrados

Codificamos las etiquetas como 0 y 1 y ajustamos una recta. Después añadimos cuatro puntos
**correctamente etiquetados** muy a la derecha: son evidencia adicional y fácil a favor de
la clase 1, justo lo que un clasificador debería agradecer.

In [ ]:
def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))


def ajusta_logistica_newton(A, t, n_iter=100):
    """Newton/IRLS. A ya incluye la columna de unos."""
    w = np.zeros(A.shape[1])
    for _ in range(n_iter):
        y = sigmoide(A @ w)
        R = y * (1 - y) + 1e-9
        H = A.T @ (A * R[:, None]) + 1e-10 * np.eye(A.shape[1])
        w = w + np.linalg.solve(H, A.T @ (t - y))
    return w


rng = np.random.default_rng(5)
x = np.r_[rng.normal(2.0, 0.8, 20), rng.normal(5.0, 0.8, 20)]
t = np.r_[np.zeros(20), np.ones(20)]
xx = np.linspace(0, 16, 400)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.5))
for ax, extra, titulo in [(axes[0], None, "datos bien separados"),
                          (axes[1], 15.0, "se añaden 4 puntos lejanos (etiqueta 1)")]:
    xa = x if extra is None else np.r_[x, [extra] * 4]
    ta = t if extra is None else np.r_[t, [1.0] * 4]
    A = np.c_[np.ones_like(xa), xa]

    b = np.linalg.lstsq(A, ta, rcond=None)[0]
    w = ajusta_logistica_newton(A, ta)
    ax.plot(xx, b[0] + b[1] * xx, label="mínimos cuadrados")
    ax.plot(xx, sigmoide(w[0] + w[1] * xx), color="firebrick", lw=2, label="logística")
    ax.scatter(xa, ta, s=26, facecolors="none", edgecolors="gray", lw=1.2)
    ax.axhline(0.5, color="gray", ls=":", lw=0.9)
    ax.set_ylim(-0.45, 1.45); ax.set_xlim(0, 16); ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("$x$"); ax.set_ylabel("$t$"); ax.legend(fontsize=8, loc="center left")

    corte_ls = (0.5 - b[0]) / b[1]
    corte_lr = -w[0] / w[1]
    print(f"{titulo:45s}  frontera MC = {corte_ls:5.2f}   frontera logística = {corte_lr:5.2f}")
fig.tight_layout(); plt.show()

La frontera de los mínimos cuadrados **se corre** al añadir evidencia correcta; la logística
casi no se mueve. El motivo: la pérdida cuadrática castiga *acertar demasiado*. Un punto en
$x=15$ con $t=1$ produce un residuo grande porque la recta predice ahí un valor mayor que 1,
así que los mínimos cuadrados inclinan la recta para reducirlo.

Y además la recta predice valores fuera de $[0,1]$, que no son probabilidades:

In [ ]:
A = np.c_[np.ones_like(x), x]
b = np.linalg.lstsq(A, t, rcond=None)[0]
pred = b[0] + b[1] * np.array([0.0, 1.0, 8.0, 12.0])
print("predicciones de mínimos cuadrados en x = 0, 1, 8, 12:", pred.round(3))

## 2. La sigmoide, los odds y el logit

$$\sigma(z) = \frac{1}{1+e^{-z}}, \qquad \sigma'(z) = \sigma(z)(1-\sigma(z))$$

Verificamos la identidad de la derivada contra una diferencia finita: es la que hace que
todos los gradientes de esta clase colapsen a algo corto.

In [ ]:
z = np.linspace(-6, 6, 9)
h = 1e-6
num = (sigmoide(z + h) - sigmoide(z - h)) / (2 * h)
ana = sigmoide(z) * (1 - sigmoide(z))
print("máxima discrepancia derivada analítica vs numérica:", np.abs(num - ana).max())

El modelo es **lineal en los log-odds**:

$$\log\frac{P(t=1\mid\mathbf{x})}{P(t=0\mid\mathbf{x})} = \mathbf{w}^{\top}\mathbf{x}$$

Tres escalas de lo mismo. Solo la tercera es una recta.

In [ ]:
xs = np.linspace(-4, 4, 400)
p = sigmoide(1.5 * xs)
fig, ax = plt.subplots(1, 3, figsize=(10.5, 3.0))
ax[0].plot(xs, p, color="firebrick", lw=2); ax[0].set_ylabel("probabilidad $p$")
ax[1].plot(xs, p / (1 - p), lw=2); ax[1].set_yscale("log"); ax[1].set_ylabel("odds $p/(1-p)$")
ax[2].plot(xs, np.log(p / (1 - p)), color="green", lw=2)
ax[2].set_ylabel(r"log-odds $\log\frac{p}{1-p}$")
for a, ti in zip(ax, ["$p=\\sigma(w x)$", "odds", "logit: recta"]):
    a.set_xlabel("$x$"); a.set_title(ti, fontsize=10)
fig.tight_layout(); plt.show()

## 3. Verosimilitud, pérdida y gradiente

Modelo de Bernoulli para una observación, con $y_n = \sigma(\mathbf{w}^{\top}\mathbf{x}_n)$:

$$p(t_n \mid \mathbf{x}_n,\mathbf{w}) = y_n^{\,t_n}(1-y_n)^{1-t_n}$$

Menos el logaritmo de la verosimilitud del conjunto completo es la **entropía cruzada**:

$$E(\mathbf{w}) = -\sum_{n=1}^{N}\big[t_n\ln y_n + (1-t_n)\ln(1-y_n)\big],
\qquad \nabla E(\mathbf{w}) = \mathbf{X}^{\top}(\mathbf{y}-\mathbf{t})$$

El gradiente es *(predicción $-$ etiqueta) $\times$ característica*, exactamente la misma
forma que en mínimos cuadrados.

In [ ]:
def nll(w, X, t):
    """Entropía cruzada. Estable numéricamente vía logaddexp."""
    z = X @ w
    # -[t*log(sigma(z)) + (1-t)*log(1-sigma(z))]  =  log(1+exp(z)) - t*z
    return np.sum(np.logaddexp(0.0, z) - t * z)


def grad_nll(w, X, t):
    return X.T @ (sigmoide(X @ w) - t)


def hess_nll(w, X):
    y = sigmoide(X @ w)
    R = y * (1 - y)
    return X.T @ (X * R[:, None])


# verificación del gradiente contra diferencias finitas centradas
rng = np.random.default_rng(0)
Xv = np.c_[np.ones(50), rng.normal(0, 1, (50, 3))]
tv = (rng.uniform(size=50) < sigmoide(Xv @ np.array([0.3, 1.0, -0.7, 0.5]))).astype(float)
w0 = rng.normal(0, 0.5, 4)

g_ana = grad_nll(w0, Xv, tv)
g_num = np.array([(nll(w0 + h * e, Xv, tv) - nll(w0 - h * e, Xv, tv)) / (2 * h)
                  for e in np.eye(4)])
print("gradiente analítico :", g_ana)
print("gradiente numérico  :", g_num)
print("máxima discrepancia :", np.abs(g_ana - g_num).max())

## 4. Convexidad

La hessiana es $\nabla^2 E = \mathbf{X}^{\top}\mathbf{R}\mathbf{X}$ con
$R_{nn} = y_n(1-y_n) > 0$, así que es semidefinida positiva **en todo punto**: el problema
es convexo, hay un solo mínimo global y la inicialización da igual.

Comparémoslo con lo que pasaría si usáramos error cuadrático sobre la salida de la sigmoide.

In [ ]:
def hess_sq(w, X, t):
    """Hessiana de sum (sigma(Xw) - t)^2."""
    y = sigmoide(X @ w)
    s = y * (1 - y)
    d2 = 2 * s ** 2 + 2 * (y - t) * s * (1 - 2 * y)
    return X.T @ (X * d2[:, None])


rng = np.random.default_rng(2)
n = 40
xs = rng.normal(0, 1.5, n)
ts = (rng.uniform(size=n) < sigmoide(1.2 * xs + 0.4)).astype(float)
Xc = np.c_[np.ones(n), xs]

rej = np.linspace(-6, 6, 45)
neg_nll = neg_sq = 0
for a in rej:
    for b_ in rej:
        w = np.array([a, b_])
        neg_nll += np.linalg.eigvalsh(hess_nll(w, Xc)).min() < -1e-9
        neg_sq += np.linalg.eigvalsh(hess_sq(w, Xc, ts)).min() < -1e-9

tot = rej.size ** 2
print(f"puntos de la rejilla con hessiana indefinida (no convexo):")
print(f"  entropía cruzada    : {100 * neg_nll / tot:5.1f}%")
print(f"  error cuadrático    : {100 * neg_sq / tot:5.1f}%")

## 5. Ajuste: gradiente frente a Newton/IRLS

Cada paso de Newton es un ajuste de **mínimos cuadrados ponderados** con pesos
$R_{nn}=y_n(1-y_n)$, máximos en $y_n=0.5$: el algoritmo presta más atención a los puntos
sobre los que está más indeciso. De ahí el nombre IRLS.

$$\mathbf{w}^{\text{nuevo}} = \mathbf{w} - (\mathbf{X}^{\top}\mathbf{R}\mathbf{X})^{-1}\mathbf{X}^{\top}(\mathbf{y}-\mathbf{t})$$

In [ ]:
def ajusta(X, t, metodo="newton", n_iter=25, lr=1.0):
    w = np.zeros(X.shape[1])
    hist = [nll(w, X, t)]
    for _ in range(n_iter):
        g = grad_nll(w, X, t)
        if metodo == "newton":
            w = w - np.linalg.solve(hess_nll(w, X) + 1e-10 * np.eye(X.shape[1]), g)
        else:
            w = w - lr / X.shape[0] * g
        hist.append(nll(w, X, t))
    return w, np.array(hist)


rng = np.random.default_rng(4)
N, p = 300, 5
Xr = rng.normal(0, 1, (N, p))
w_verdad = rng.normal(0, 1.2, p)
tr = (rng.uniform(size=N) < sigmoide(Xr @ w_verdad)).astype(float)
Xa = np.c_[np.ones(N), Xr]

w_new, h_new = ajusta(Xa, tr, "newton")
_, h_g10 = ajusta(Xa, tr, "gd", lr=10.0)
_, h_g1 = ajusta(Xa, tr, "gd", lr=1.0)
mejor = min(h_new.min(), h_g10.min(), h_g1.min())

plt.semilogy(h_new - mejor + 1e-15, "o-", color="firebrick", label="Newton / IRLS")
plt.semilogy(h_g10 - mejor + 1e-15, "s-", label=r"gradiente, $\alpha=10$")
plt.semilogy(h_g1 - mejor + 1e-15, "^-", color="green", label=r"gradiente, $\alpha=1$")
plt.xlabel("iteración"); plt.ylabel(r"$E(\mathbf{w}) - E(\mathbf{w}^\star)$")
plt.legend(fontsize=9); plt.show()

Newton llega a precisión de máquina en unas cinco iteraciones (convergencia cuadrática:
el número de dígitos correctos se duplica en cada paso) y **no tiene tasa de aprendizaje que
ajustar**. El descenso por gradiente sigue avanzando después de 25, y su velocidad depende
por completo del $\alpha$ que uno haya elegido.

Comprobamos contra `sklearn`, pidiéndole que **no** regularice:

In [ ]:
from sklearn.linear_model import LogisticRegression

mdl = LogisticRegression(C=1e12, tol=1e-12, max_iter=100000).fit(Xr, tr)
w_sk = np.r_[mdl.intercept_, mdl.coef_.ravel()]

print("propio  :", w_new)
print("sklearn :", w_sk)
print("máxima discrepancia:", np.abs(w_new - w_sk).max())

## 6. Datos separables: el estimador no existe

Si un hiperplano separa las clases perfectamente, escalar $\mathbf{w}$ por 10 deja la frontera
donde está pero hace *todas* las predicciones más seguras, así que la verosimilitud **aumenta**.
Nunca hay razón para detenerse: el óptimo está en el infinito.

In [ ]:
rng = np.random.default_rng(3)
X0 = rng.multivariate_normal([-1.6, 0], 0.3 * np.eye(2), 25)
X1 = rng.multivariate_normal([1.6, 0], 0.3 * np.eye(2), 25)
Xs = np.vstack([X0, X1])
ts_sep = np.r_[np.zeros(25), np.ones(25)]
As = np.c_[np.ones(50), Xs]

w = np.zeros(3)
normas, perdidas = [], []
for _ in range(4000):
    w = w - 0.05 * grad_nll(w, As, ts_sep)
    normas.append(np.linalg.norm(w[1:]))
    perdidas.append(nll(w, As, ts_sep))

fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
ax[0].scatter(*X0.T, s=22); ax[0].scatter(*X1.T, s=22, color="firebrick", marker="^")
ax[0].axvline(0, color="gray", ls="--"); ax[0].set_title("datos separables", fontsize=10)
ax[1].plot(normas, color="firebrick"); ax[1].set_ylabel(r"$\|\mathbf{w}\|_2$")
ax[1].set_title("la norma diverge", fontsize=10)
ax[2].plot(perdidas); ax[2].set_ylabel("entropía cruzada")
ax[2].set_title(r"$E \to 0$, nunca alcanzado", fontsize=10)
for a in ax[1:]:
    a.set_xlabel("iteración")
fig.tight_layout(); plt.show()

print(f"tras 4000 pasos:  ||w|| = {normas[-1]:.2f}   E(w) = {perdidas[-1]:.2e}")
print("¿se estabilizó la norma?  crecimiento en los últimos 500 pasos:",
      round(normas[-1] - normas[-500], 3))

La norma sigue creciendo y la pérdida sigue bajando: no hay convergencia. El software
**no da error**, simplemente agota sus iteraciones y devuelve coeficientes enormes.

Con una penalización, la pérdida sigue bajando pero la penalización sube, y la suma tiene un
mínimo en norma finita:

In [ ]:
for lam in [0.0, 0.01, 0.1, 1.0]:
    w = np.zeros(3)
    for _ in range(4000):
        g = grad_nll(w, As, ts_sep) + 2 * lam * np.r_[0.0, w[1:]]   # sin penalizar el sesgo
        w = w - 0.05 * g
    print(f"lambda = {lam:5.2f}  ->  ||w|| = {np.linalg.norm(w[1:]):8.2f}")

## 7. Regularización, validación cruzada e interpretación

Datos de cáncer de mama de sklearn: $N = 569$, $p = 30$ características estandarizadas.
La penalización es exactamente la de la clase anterior, montada sobre otra pérdida.

**Aviso sobre `sklearn`:** `LogisticRegression` aplica $L_2$ con `C=1` **por defecto**.
Si uno quiere máxima verosimilitud pura, hay que pasar un `C` enorme.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = load_breast_cancer()
Xb, yb, nombres = d.data, d.target, list(d.feature_names)
print("X:", Xb.shape, " positivos:", yb.mean().round(3))

Cs = np.logspace(-3, 2, 30)
lams = 1.0 / Cs
kf = KFold(10, shuffle=True, random_state=0)

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.6))
elegidos = {}
for a, pen in zip(ax, ["l2", "l1"]):
    medias, ses = [], []
    for C in Cs:
        # el escalado va DENTRO del pliegue: si no, hay fuga de información
        mdl = make_pipeline(StandardScaler(),
                            LogisticRegression(penalty=pen, C=C, solver="liblinear",
                                               max_iter=10000))
        s = cross_val_score(mdl, Xb, yb, cv=kf, scoring="neg_log_loss")
        medias.append(-s.mean()); ses.append(s.std(ddof=1) / np.sqrt(10))
    medias, ses = np.array(medias), np.array(ses)

    i = int(np.argmin(medias))
    dentro = np.where(medias <= medias[i] + ses[i])[0]
    # lambda = 1/C decrece con C: el lambda mayor admisible es el C menor
    i1se = int(dentro.min())
    elegidos[pen] = (Cs[i], Cs[i1se])

    a.errorbar(lams, medias, yerr=ses, color="firebrick", ecolor="#c99", capsize=2)
    a.set_xscale("log")
    a.axvline(1 / Cs[i], ls="--", label=r"$\lambda_{min}$")
    a.axvline(1 / Cs[i1se], ls=":", color="green", label="un error estándar")
    a.set_xlabel(r"$\lambda = 1/C$"); a.set_ylabel("log-loss de CV")
    a.set_title(f"penalización {pen.upper()}", fontsize=10); a.legend(fontsize=8)
fig.tight_layout(); plt.show()

for pen, (c_min, c_1se) in elegidos.items():
    print(f"{pen}:  C_min = {c_min:8.4f} (λ={1/c_min:8.3f})   "
          f"C_1se = {c_1se:8.4f} (λ={1/c_1se:8.3f})")

Cuántas características sobreviven con cada penalización, usando el $\lambda$ de la regla
de un error estándar:

In [ ]:
Xs_std = StandardScaler().fit_transform(Xb)
for pen in ["l2", "l1"]:
    C = elegidos[pen][1]
    m = LogisticRegression(penalty=pen, C=C, solver="liblinear",
                           max_iter=10000).fit(Xs_std, yb)
    nz = np.flatnonzero(m.coef_.ravel())
    print(f"{pen}: {len(nz):2d}/30 coeficientes no nulos")
    if pen == "l1":
        sel = sorted(zip(np.abs(m.coef_.ravel()[nz]), [str(nombres[j]) for j in nz]),
                     reverse=True)
        print("     seleccionadas, por |w| decreciente:")
        for mag, nom in sel:
            print(f"       {nom:28s} |w| = {mag:.3f}")

### Odds ratios

Aumentar $x_j$ en una unidad (aquí, una desviación estándar) multiplica los **odds** por
$e^{w_j}$. Esta es la frase que se pone en un paper, y la razón por la que la regresión
logística sigue siendo el modelo por defecto en epidemiología.

Cuidado: es un *odds ratio*, no un *risk ratio*. Se parecen cuando el evento es raro y
difieren bastante cuando es frecuente.

In [ ]:
m = LogisticRegression(penalty="l2", C=elegidos["l2"][1], max_iter=10000).fit(Xs_std, yb)
coef = m.coef_.ravel()
tabla = (pd.DataFrame({"w_j": coef, "odds ratio  exp(w_j)": np.exp(coef)}, index=nombres)
         .reindex(np.abs(coef).argsort()[::-1]).head(8))
tabla.round(3)

## 8. ¿Son honestas las probabilidades?

Un modelo puede ordenar los casos perfectamente y aun así mentir sobre las probabilidades.
El diagrama de fiabilidad agrupa las predicciones y pregunta, dentro de cada grupo, qué
fracción resultó positiva. Sobre la diagonal significa honesto.

La exactitud y el AUC son **invariantes a cualquier transformación monótona** de los scores:
se puede estar mal calibrado y puntuar estupendamente en ambos.

In [ ]:
from sklearn.naive_bayes import GaussianNB

# predictores muy correlacionados: naive Bayes supone independencia y por tanto
# cuenta la misma evidencia varias veces, volviéndose sobreconfiado
rng = np.random.default_rng(8)
n, pdim = 4000, 12
z = rng.normal(0, 1, (n, 1))
Xk = z + rng.normal(0, 0.45, (n, pdim))
yk = (rng.uniform(size=n) < sigmoide(Xk @ np.full(pdim, 0.55))).astype(int)
Xtr, Xte, ytr, yte = train_test_split(Xk, yk, test_size=0.5, random_state=0)

from sklearn.metrics import roc_auc_score, brier_score_loss

plt.plot([0, 1], [0, 1], ls="--", color="gray", label="calibración perfecta")
for nombre, mdl, mk in [("regresión logística", LogisticRegression(max_iter=5000), "o"),
                        ("naive Bayes gaussiano", GaussianNB(), "s")]:
    prob = mdl.fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    bins = np.linspace(0, 1, 11)
    idx = np.digitize(prob, bins) - 1
    xs_, ys_ = [], []
    for b_ in range(10):
        msk = idx == b_
        if msk.sum() > 10:
            xs_.append(prob[msk].mean()); ys_.append(yte[msk].mean())
    plt.plot(xs_, ys_, mk + "-", label=nombre, ms=5)
    print(f"{nombre:24s}  AUC = {roc_auc_score(yte, prob):.3f}   "
          f"Brier = {brier_score_loss(yte, prob):.4f}")
plt.xlabel("probabilidad predicha"); plt.ylabel("frecuencia observada")
plt.legend(fontsize=8); plt.gca().set_aspect("equal"); plt.show()

Fíjese en los números impresos: los dos modelos tienen un AUC casi idéntico —ordenan igual
de bien— pero el Brier score, que sí penaliza la calibración, los separa.

## 9. Softmax multiclase

$$P(t=k\mid\mathbf{x}) = \frac{\exp(\mathbf{w}_k^{\top}\mathbf{x})}{\sum_j \exp(\mathbf{w}_j^{\top}\mathbf{x})},
\qquad \frac{\partial E}{\partial \mathbf{w}_k} = \sum_n (y_{nk}-t_{nk})\mathbf{x}_n$$

El mismo gradiente por tercera vez. Y con $K=2$ el softmax se reduce exactamente a la
sigmoide: lo comprobamos.

In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)   # nunca exponenciar scores crudos: desborda
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)


w2 = np.array([0.7, -1.3, 0.4])
Xd = np.c_[np.ones(6), np.random.default_rng(1).normal(0, 1, (6, 2))]
# softmax con dos clases: w_1 = 0, w_2 = w  ->  debe dar la sigmoide de Xw
P = softmax(np.c_[np.zeros(6), Xd @ w2])
print("softmax clase 2 :", P[:, 1].round(6))
print("sigmoide(Xw)    :", sigmoide(Xd @ w2).round(6))
print("máxima discrepancia:", np.abs(P[:, 1] - sigmoide(Xd @ w2)).max())

In [ ]:
rng = np.random.default_rng(6)
mus = [[0, 0], [3.4, 0.6], [1.6, 3.2]]
Xs3 = [rng.multivariate_normal(m, 0.55 * np.eye(2), 60) for m in mus]
X3 = np.vstack(Xs3); y3 = np.repeat([0, 1, 2], 60)
mdl3 = LogisticRegression(C=10.0, max_iter=5000).fit(X3, y3)

gx, gy = np.meshgrid(np.linspace(X3[:, 0].min() - 1, X3[:, 0].max() + 1, 300),
                     np.linspace(X3[:, 1].min() - 1, X3[:, 1].max() + 1, 300))
G = np.c_[gx.ravel(), gy.ravel()]

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.8))
ax[0].contourf(gx, gy, mdl3.predict(G).reshape(gx.shape), levels=[-.5, .5, 1.5, 2.5],
               colors=["#dbe6fb", "#f6dede", "#dff0d8"])
for Xi, c, mk in zip(Xs3, ["tab:blue", "firebrick", "green"], ["o", "^", "s"]):
    ax[0].scatter(*Xi.T, s=16, color=c, marker=mk)
ax[0].set_title(r"$\arg\max_k P(y=k\mid\mathbf{x})$", fontsize=10)

cs = ax[1].contourf(gx, gy, mdl3.predict_proba(G).max(1).reshape(gx.shape), levels=20)
fig.colorbar(cs, ax=ax[1], label=r"$\max_k P(y=k\mid\mathbf{x})$")
ax[1].set_title("la confianza cae en las fronteras", fontsize=10)
for a in ax:
    a.set_xlabel("$x_1$"); a.set_ylabel("$x_2$"); a.grid(False)
fig.tight_layout(); plt.show()

La banda oscura del panel derecho es donde el modelo no sabe. En un despliegue clínico ahí
es donde uno se abstiene y escala a un humano — algo que un modelo que solo entrega
etiquetas no puede hacer.

## 10. Ejercicios

1. **Reducción a la sigmoide.** Demuestre en papel que con $K=2$ el softmax da
   $\sigma((\mathbf{w}_2-\mathbf{w}_1)^{\top}\mathbf{x})$, y explique por qué eso implica que los
   parámetros del softmax están sobredeterminados. ¿Qué hace la regularización con esa
   ambigüedad?

2. **Fronteras curvas.** Añada las características $x_1^2$, $x_2^2$ y $x_1x_2$ al problema
   de dos clases de la sección 6 y grafique la frontera resultante en el espacio original.
   ¿Sigue siendo lineal el modelo? ¿En qué sentido?

3. **Clases desbalanceadas.** Submuestree la clase positiva del conjunto de cáncer hasta un
   5%. Compare exactitud, AUC y log-loss contra la línea base que predice siempre la clase
   mayoritaria. ¿Cuál de las tres métricas detecta el problema?

4. **IRLS a mano.** Implemente el paso de Newton en su forma de mínimos cuadrados ponderados,
   usando la respuesta ajustada $\mathbf{z} = \mathbf{X}\mathbf{w} + \mathbf{R}^{-1}(\mathbf{t}-\mathbf{y})$,
   y verifique que da exactamente lo mismo que la forma de la sección 5.

5. **Umbral de decisión.** El umbral 0.5 solo es óptimo si los dos errores cuestan lo mismo.
   Suponga que un falso negativo cuesta 10 veces más que un falso positivo y derive el
   umbral óptimo. Aplíquelo al modelo de la sección 7 y compare las matrices de confusión.

6. **Calibración.** Ajuste `CalibratedClassifierCV` sobre el naive Bayes de la sección 8
   (con datos de validación separados, nunca los de entrenamiento) y rehaga el diagrama de
   fiabilidad. ¿Mejora el Brier score? ¿Cambia el AUC? Explique por qué.